# Sample4Geo — XAI Analysis

**Course:** Hacettepe University — Computer Vision  
**Topic:** Explainable AI (XAI) on Cross-View Geo-Localization  
**Base paper:** Sample4Geo (ICCV 2023)  

**Objective:**  
Apply two XAI methods (GradCAM and Occlusion Sensitivity) to a pretrained Sample4Geo model and analyze what image regions the model attends to when matching drone/street-level views to satellite images.

**Datasets:**
- University-1652 (drone ↔ satellite)
- VIGOR (street panorama ↔ satellite, same-area and cross-area splits)

**Success criteria:**
- Baseline evaluation metrics reproduced for both datasets
- GradCAM and Occlusion Sensitivity heatmaps generated for successful and failed matches
- Faithfulness curves computed for Occlusion Sensitivity
- Visual comparison between the two XAI methods


## 0. Environment Setup

Run this section first. It mounts Google Drive, clones the repo, installs dependencies, and sets all paths.

> **Note:** The dataset lives at `MyDrive/vision-datasets/`. The code repo is cloned from GitHub.


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_ROOT       = '/content/drive/MyDrive/vision-datasets'
U1652_ROOT       = f'{DRIVE_ROOT}/University-Release'
VIGOR_ROOT       = f'{DRIVE_ROOT}/VIGOR'
REPO_DIR         = '/content/Sample4Geo'
PRETRAINED_DIR   = f'{REPO_DIR}/pretrained'
RESULTS_DIR      = f'{REPO_DIR}/xai_results'

# Pretrained weight paths
CKPT_U1652       = f'{PRETRAINED_DIR}/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth'
CKPT_VIGOR_SAME  = f'{PRETRAINED_DIR}/vigor_same/convnext_base.fb_in22k_ft_in1k_384/weights_e40_0.7786.pth'
CKPT_VIGOR_CROSS = f'{PRETRAINED_DIR}/vigor_cross/convnext_base.fb_in22k_ft_in1k_384/weights_e40_0.6109.pth'

print('Drive root exists:', os.path.exists(DRIVE_ROOT))
print('U1652 root exists:', os.path.exists(U1652_ROOT))
print('VIGOR root exists:', os.path.exists(VIGOR_ROOT))

Drive root exists: True
U1652 root exists: True
VIGOR root exists: True


In [4]:
# Clone repo and install dependencies
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/halisyucel/Sample4Geo.git', REPO_DIR], check=True)
else:
    print('Repo already cloned, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

%pip install -q timm albumentations opencv-python-headless

Repo already cloned, pulling latest...


In [5]:
# Add repo to path and verify GPU
import sys
sys.path.insert(0, REPO_DIR)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

os.makedirs(RESULTS_DIR, exist_ok=True)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 1. Download Pretrained Weights

Weights are hosted on Google Drive (Sample4Geo authors).  
Download once and save to `pretrained/` inside the repo directory.

Source: https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA

| Dataset | File | Reported R@1 |
|---|---|---|
| University-1652 | `weights_e1_0.9515.pth` | 95.15% |
| VIGOR same-area | `weights_e40_0.7786.pth` | 77.86% |
| VIGOR cross-area | `weights_e40_0.6109.pth` | 61.09% |


In [6]:
# Download weights using gdown
# Run once; skip if already downloaded.
%pip install -q gdown
import gdown

weights = [
    # (gdrive_file_id, local_path)
    # Fill in file IDs from the Google Drive folder above after opening it
    # Example: ('1ABC...XYZ', f'{PRETRAINED_DIR}/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth'),
]

for file_id, dest in weights:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        gdown.download(id=file_id, output=dest, quiet=False)
    else:
        print(f'Already exists: {dest}')

# Verify
for _, dest in weights:
    print(os.path.exists(dest), dest)

## 2. Dataset Paths

Datasets are read directly from Google Drive — no copying needed.

**University-1652 structure (from Drive):**
```
University-Release/
├── train/
│   ├── drone/, satellite/, google/, street/
└── test/
    ├── query_drone/, gallery_satellite/
    ├── query_satellite/, gallery_drone/
    └── ...
```

**VIGOR structure (from Drive):**
```
VIGOR/
├── Chicago/, NewYork/, SanFrancisco/, Seattle/
│   ├── panorama/   ← street-view (ground)
│   └── satellite/  ← satellite tiles
└── splits/
    └── {city}/
        ├── satellite_list.txt
        ├── pano_label_balanced.txt
        ├── same_area_balanced_train.txt
        └── same_area_balanced_test.txt
```

> **Important:** The `VigorDataset` class in the original code expects `data_folder/ground/{city}/` but the Drive has `panorama/` instead.  
> We create symlinks below to bridge this.


In [7]:
# Create symlinks so VigorDataset can find ground/ -> panorama/
# The dataset class hardcodes data_folder/ground/{city}/ paths.
cities = ['Chicago', 'NewYork', 'SanFrancisco', 'Seattle']

for city in cities:
    src = os.path.join(VIGOR_ROOT, city, 'panorama')
    dst = os.path.join(VIGOR_ROOT, city, 'ground')
    # Also the dataset reads data_folder/satellite/{city}/ — this is already correct
    # but it reads data_folder/ground/{city}/ for panoramas
    # We need a different approach: patch the dataset or use a wrapper data_folder
    print(f'{city}: panorama exists = {os.path.exists(src)}')

# Build a VIGOR data_folder that matches what VigorDataset expects:
# data_folder/ground/{city}/ -> symlink to panorama
# data_folder/satellite/{city}/ -> symlink to satellite
# data_folder/splits/{city}/ -> symlink to splits
VIGOR_DATA = '/content/vigor_data'
os.makedirs(VIGOR_DATA, exist_ok=True)

for city in cities:
    # ground symlink (panorama -> ground)
    ground_dir = os.path.join(VIGOR_DATA, 'ground', city)
    sat_dir    = os.path.join(VIGOR_DATA, 'satellite', city)
    split_dir  = os.path.join(VIGOR_DATA, 'splits', city)

    os.makedirs(os.path.dirname(ground_dir), exist_ok=True)
    os.makedirs(os.path.dirname(sat_dir),    exist_ok=True)
    os.makedirs(os.path.dirname(split_dir),  exist_ok=True)

    if not os.path.exists(ground_dir):
        os.symlink(os.path.join(VIGOR_ROOT, city, 'panorama'), ground_dir)
    if not os.path.exists(sat_dir):
        os.symlink(os.path.join(VIGOR_ROOT, city, 'satellite'), sat_dir)
    if not os.path.exists(split_dir):
        os.symlink(os.path.join(VIGOR_ROOT, 'splits', city), split_dir)

    print(f'{city}: ground={os.path.exists(ground_dir)}, sat={os.path.exists(sat_dir)}, splits={os.path.exists(split_dir)}')

Chicago: panorama exists = True
NewYork: panorama exists = True
SanFrancisco: panorama exists = True
Seattle: panorama exists = True
Chicago: ground=True, sat=True, splits=True
NewYork: ground=True, sat=True, splits=True
SanFrancisco: ground=True, sat=True, splits=True
Seattle: ground=True, sat=True, splits=True


## 3. Baseline Evaluation — University-1652

Task: drone → satellite retrieval (D2S).  
Model: ConvNeXt-Base pretrained on University-1652.  
Expected R@1 ≈ 95.15% (paper). Our earlier Mac run got 92.67%.


In [8]:
from dataclasses import dataclass
from torch.utils.data import DataLoader
from sample4geo.dataset.university import U1652DatasetEval, get_transforms
from sample4geo.evaluate.university import evaluate
from sample4geo.model import TimmModel

@dataclass
class U1652Config:
    model: str            = 'convnext_base.fb_in22k_ft_in1k_384'
    img_size: int         = 384
    batch_size: int       = 128
    verbose: bool         = True
    gpu_ids: tuple        = (0,)
    normalize_features: bool = True
    eval_gallery_n: int   = -1
    dataset: str          = 'U1652-D2S'
    num_workers: int      = 4
    device: str           = device
    checkpoint_start: str = CKPT_U1652

cfg_u = U1652Config()
cfg_u.query_folder_test   = f'{U1652_ROOT}/test/query_drone'
cfg_u.gallery_folder_test = f'{U1652_ROOT}/test/gallery_satellite'

print('Config:', cfg_u)

Config: U1652Config(model='convnext_base.fb_in22k_ft_in1k_384', img_size=384, batch_size=128, verbose=True, gpu_ids=(0,), normalize_features=True, eval_gallery_n=-1, dataset='U1652-D2S', num_workers=4, device='cuda', checkpoint_start='/content/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth')


In [9]:
# Load model
model_u = TimmModel(cfg_u.model, pretrained=True, img_size=cfg_u.img_size)
data_config = model_u.get_config()
mean, std = data_config['mean'], data_config['std']
img_size = (cfg_u.img_size, cfg_u.img_size)

state_dict = torch.load(cfg_u.checkpoint_start, map_location='cpu')
model_u.load_state_dict(state_dict, strict=False)
model_u = model_u.to(device).eval()
print('Model loaded.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth'

In [ ]:
# Build dataloaders
val_transforms, _, _ = get_transforms(img_size, mean=mean, std=std)

query_dataset = U1652DatasetEval(
    data_folder=cfg_u.query_folder_test, mode='query', transforms=val_transforms)
gallery_dataset = U1652DatasetEval(
    data_folder=cfg_u.gallery_folder_test, mode='gallery',
    transforms=val_transforms, sample_ids=query_dataset.get_sample_ids(), gallery_n=-1)

query_loader = DataLoader(
    query_dataset, batch_size=cfg_u.batch_size,
    num_workers=cfg_u.num_workers, shuffle=False, pin_memory=True)
gallery_loader = DataLoader(
    gallery_dataset, batch_size=cfg_u.batch_size,
    num_workers=cfg_u.num_workers, shuffle=False, pin_memory=True)

print(f'Query images:   {len(query_dataset)}')
print(f'Gallery images: {len(gallery_dataset)}')

In [ ]:
# Run evaluation
r1_u, result_str_u = evaluate(
    config=cfg_u,
    model=model_u,
    query_loader=query_loader,
    gallery_loader=gallery_loader,
    ranks=[1, 5, 10],
    step_size=1000,
    cleanup=True,
)
print('\n=== University-1652 Results ===')
print(result_str_u)

## 4. Baseline Evaluation — VIGOR

Two evaluation modes:
- **Same-area:** train/test cities overlap geographically (harder)
- **Cross-area:** train on NYC+Seattle, test on Chicago+SanFrancisco

Task: street-view panorama → satellite retrieval.


In [ ]:
from sample4geo.dataset.vigor import VigorDatasetEval
from sample4geo.transforms import get_transforms_val
from sample4geo.evaluate.vigor import evaluate as evaluate_vigor

@dataclass
class VigorConfig:
    model: str            = 'convnext_base.fb_in22k_ft_in1k_384'
    img_size: int         = 384
    batch_size: int       = 128
    verbose: bool         = True
    gpu_ids: tuple        = (0,)
    normalize_features: bool = True
    data_folder: str      = VIGOR_DATA
    ground_cutting: int   = 0
    num_workers: int      = 4
    device: str           = device
    same_area: bool       = True
    checkpoint_start: str = CKPT_VIGOR_SAME

def run_vigor_eval(same_area: bool):
    cfg = VigorConfig(
        same_area=same_area,
        checkpoint_start=CKPT_VIGOR_SAME if same_area else CKPT_VIGOR_CROSS,
    )

    model_v = TimmModel(cfg.model, pretrained=True, img_size=cfg.img_size)
    data_config = model_v.get_config()
    mean_v, std_v = data_config['mean'], data_config['std']
    img_size_v = cfg.img_size

    state_dict = torch.load(cfg.checkpoint_start, map_location='cpu')
    model_v.load_state_dict(state_dict, strict=False)
    model_v = model_v.to(device).eval()

    image_size_sat = (img_size_v, img_size_v)
    new_width = img_size_v * 2
    new_height = int(((1024 - 2 * cfg.ground_cutting) / 2048) * new_width)
    img_size_ground = (new_height, new_width)

    sat_transforms, ground_transforms = get_transforms_val(
        image_size_sat, img_size_ground, mean=mean_v, std=std_v,
        ground_cutting=cfg.ground_cutting)

    ref_dataset = VigorDatasetEval(
        data_folder=cfg.data_folder, split='test', img_type='reference',
        same_area=cfg.same_area, transforms=sat_transforms)
    q_dataset = VigorDatasetEval(
        data_folder=cfg.data_folder, split='test', img_type='query',
        same_area=cfg.same_area, transforms=ground_transforms)

    ref_loader = DataLoader(
        ref_dataset, batch_size=cfg.batch_size,
        num_workers=cfg.num_workers, shuffle=False, pin_memory=True)
    q_loader = DataLoader(
        q_dataset, batch_size=cfg.batch_size,
        num_workers=cfg.num_workers, shuffle=False, pin_memory=True)

    tag = 'VIGOR Same' if same_area else 'VIGOR Cross'
    print(f'\n{"-"*30}[{tag}]{"-"*30}')
    print(f'Query: {len(q_dataset)}  Reference: {len(ref_dataset)}')

    r1, result_str = evaluate_vigor(
        config=cfg, model=model_v,
        reference_dataloader=ref_loader,
        query_dataloader=q_loader,
        ranks=[1, 5, 10], step_size=1000, cleanup=True)

    return r1, result_str

In [ ]:
r1_same, result_same = run_vigor_eval(same_area=True)
print('VIGOR Same-Area:', result_same)

In [ ]:
r1_cross, result_cross = run_vigor_eval(same_area=False)
print('VIGOR Cross-Area:', result_cross)

## 5. XAI Setup

Two complementary methods:

| Method | Type | How it works |
|---|---|---|
| **GradCAM** | White-box, gradient-based | Backpropagates cosine similarity gradient through last conv layer; weights activations by mean gradient |
| **Occlusion Sensitivity** | Black-box, perturbation-based | Slides a patch over the image; measures cosine similarity drop at each position |

GradCAM is fast (~1 forward + 1 backward). Occlusion is slow (~(H/stride)² forward passes).


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2


# ── Image loading helper ───────────────────────────────────────────────────────
def load_image(path: str, img_size: int = 384) -> torch.Tensor:
    """Load, resize and normalize an image. Returns (1, C, H, W) tensor."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR_EXACT),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return transform(image=img)['image'].unsqueeze(0)


# ── Tensor → displayable numpy ─────────────────────────────────────────────────
def tensor_to_display(t: torch.Tensor) -> np.ndarray:
    """Denormalize and convert to uint8 HWC."""
    img = t.squeeze().cpu().numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)


print('Helpers ready.')

### 5.1 GradCAM


In [ ]:
class GradCAMExtractor:
    """
    GradCAM for ConvNeXt models in Sample4Geo.
    Visualizes which regions the model focuses on for geo-localization.
    """

    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        self.target_layer = target_layer or self._find_target_layer()
        self.gradients = None
        self.activations = None
        self._hooks = []
        self._register_hooks()

    def _find_target_layer(self):
        """Find the last block of the last ConvNeXt stage."""
        target = None
        for name, module in self.model.model.named_modules():
            parts = name.split('.')
            if 'stages' in name and len(parts) == 3:
                target = module
        if target is None:
            for _, module in reversed(list(self.model.model.named_modules())):
                if isinstance(module, torch.nn.Conv2d):
                    target = module
                    break
        return target

    def _register_hooks(self):
        def fwd(module, inp, out):
            self.activations = out.detach()
        def bwd(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self._hooks.append(self.target_layer.register_forward_hook(fwd))
        self._hooks.append(self.target_layer.register_full_backward_hook(bwd))

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def generate_cam(self, image: torch.Tensor,
                     target_embedding: torch.Tensor = None) -> np.ndarray:
        self.model.zero_grad()
        embedding = self.model(image)
        if target_embedding is not None:
            score = F.cosine_similarity(embedding, target_embedding, dim=-1)
        else:
            score = embedding.norm(dim=-1).mean()
        score.backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cv2.resize(cam, (image.shape[3], image.shape[2]))

    def generate_pair_cam(self, query: torch.Tensor,
                          gallery: torch.Tensor):
        with torch.no_grad():
            gallery_emb = self.model(gallery)
        query_cam = self.generate_cam(query, gallery_emb)
        with torch.no_grad():
            query_emb = self.model(query)
        gallery_cam = self.generate_cam(gallery, query_emb)
        return query_cam, gallery_cam

    @staticmethod
    def overlay(image: torch.Tensor, cam: np.ndarray,
                alpha: float = 0.4) -> np.ndarray:
        img_np = tensor_to_display(image)
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        return cv2.addWeighted(img_np, 1 - alpha, heatmap, alpha, 0)


print('GradCAMExtractor defined.')

### 5.2 Occlusion Sensitivity


In [ ]:
class OcclusionSensitivity:
    """
    Perturbation-based XAI.
    Slides an occlusion patch over the query image and measures cosine similarity drop.
    """

    def __init__(self, model, device=None):
        self.model = model
        self.model.eval()
        self.device = device or next(model.parameters()).device

    def compute_sensitivity(self, query_image: torch.Tensor,
                             gallery_embedding: torch.Tensor,
                             patch_size: int = 64,
                             stride: int = 16,
                             occlusion_value: float = 0.0) -> np.ndarray:
        query_image = query_image.to(self.device)
        gallery_embedding = gallery_embedding.to(self.device)
        _, _, H, W = query_image.shape

        with torch.no_grad():
            base_emb = self.model(query_image)
            base_score = F.cosine_similarity(base_emb, gallery_embedding, dim=-1).item()

        importance = np.zeros((H, W), dtype=np.float32)
        counts     = np.zeros((H, W), dtype=np.float32)

        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                occ = query_image.clone()
                occ[:, :, y:y + patch_size, x:x + patch_size] = occlusion_value
                with torch.no_grad():
                    occ_emb = self.model(occ)
                    occ_score = F.cosine_similarity(occ_emb, gallery_embedding, dim=-1).item()
                drop = base_score - occ_score
                importance[y:y + patch_size, x:x + patch_size] += drop
                counts[y:y + patch_size, x:x + patch_size] += 1

        importance = importance / np.maximum(counts, 1)
        importance = np.maximum(importance, 0)
        mx = importance.max()
        if mx > 1e-8:
            importance /= mx
        return importance

    def compute_pair_sensitivity(self, query: torch.Tensor, gallery: torch.Tensor,
                                  patch_size: int = 64, stride: int = 16):
        query   = query.to(self.device)
        gallery = gallery.to(self.device)
        with torch.no_grad():
            gallery_emb = self.model(gallery)
            query_emb   = self.model(query)
        q_map = self.compute_sensitivity(query,   gallery_emb, patch_size, stride)
        g_map = self.compute_sensitivity(gallery, query_emb,   patch_size, stride)
        return q_map, g_map

    def compute_faithfulness(self, query: torch.Tensor,
                              gallery_embedding: torch.Tensor,
                              importance_map: np.ndarray,
                              steps: int = 10,
                              occlusion_value: float = 0.0):
        query = query.to(self.device)
        gallery_embedding = gallery_embedding.to(self.device)
        with torch.no_grad():
            base_emb   = self.model(query)
            base_score = F.cosine_similarity(base_emb, gallery_embedding, dim=-1).item()

        _, _, H, W = query.shape
        flat_importance = importance_map.flatten()
        sorted_indices  = np.argsort(flat_importance)[::-1]
        total_pixels    = H * W

        fractions = np.linspace(0, 1, steps + 1)
        scores = [base_score]
        for frac in fractions[1:]:
            n_mask = int(frac * total_pixels)
            mask_idx = sorted_indices[:n_mask]
            occ = query.clone()
            flat_img = occ.reshape(1, 3, -1)
            flat_img[:, :, mask_idx] = occlusion_value
            occ = flat_img.reshape_as(query)
            with torch.no_grad():
                occ_emb = self.model(occ)
                score   = F.cosine_similarity(occ_emb, gallery_embedding, dim=-1).item()
            scores.append(score)
        return fractions, np.array(scores)

    @staticmethod
    def overlay(image: torch.Tensor, importance_map: np.ndarray,
                alpha: float = 0.5) -> np.ndarray:
        img_np = tensor_to_display(image)
        heatmap = cv2.applyColorMap(np.uint8(255 * importance_map), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        return cv2.addWeighted(img_np, 1 - alpha, heatmap, alpha, 0)


print('OcclusionSensitivity defined.')

### 5.3 Visualization helper


In [ ]:
def show_gradcam(query_t, gallery_t, query_cam, gallery_cam,
                 title='', save_path=None):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    rows = [
        (query_t,   query_cam,   'Query'),
        (gallery_t, gallery_cam, 'Gallery'),
    ]
    for row_idx, (img_t, cam, label) in enumerate(rows):
        img_np = tensor_to_display(img_t)
        overlay = GradCAMExtractor.overlay(img_t, cam)
        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title(f'{label} — Original')
        axes[row_idx, 1].imshow(cam, cmap='jet')
        axes[row_idx, 1].set_title(f'{label} — GradCAM')
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title(f'{label} — Overlay')
    for ax in axes.flat:
        ax.axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def show_occlusion(query_t, gallery_t, query_map, gallery_map,
                   q_faith=None, g_faith=None, title='', save_path=None):
    n_cols = 4 if q_faith else 3
    fig, axes = plt.subplots(2, n_cols, figsize=(5 * n_cols, 10))
    rows = [
        (query_t,   query_map,   q_faith, 'Query'),
        (gallery_t, gallery_map, g_faith, 'Gallery'),
    ]
    for row_idx, (img_t, imap, faith, label) in enumerate(rows):
        img_np  = tensor_to_display(img_t)
        overlay = OcclusionSensitivity.overlay(img_t, imap)
        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title(f'{label} — Original')
        axes[row_idx, 1].imshow(imap, cmap='jet', vmin=0, vmax=1)
        axes[row_idx, 1].set_title(f'{label} — Importance Map')
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title(f'{label} — Overlay')
        if faith and n_cols == 4:
            fracs, scores = faith
            axes[row_idx, 3].plot(fracs * 100, scores, marker='o', color='crimson')
            axes[row_idx, 3].set_xlabel('Masked Pixels (%)')
            axes[row_idx, 3].set_ylabel('Cosine Similarity')
            axes[row_idx, 3].set_title(f'{label} — Faithfulness')
            axes[row_idx, 3].grid(True, alpha=0.3)
    for ax in axes.flat:
        ax.axis('off') if ax.has_data() and ax.get_xlabel() == '' else None
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print('Visualization helpers ready.')

## 6. XAI Pair Selection — University-1652

We need representative pairs:
- **Successful matches:** model's top-1 is correct  
- **Failed matches:** model's top-1 is wrong

Run this section after Section 3 (eval) so embeddings are computed.


In [ ]:
# Re-extract embeddings for pair selection
# (If Section 3 already ran and cleanup=True, we need to recompute)
from sample4geo.trainer import predict

# Reload model if needed
model_u = TimmModel(cfg_u.model, pretrained=True, img_size=cfg_u.img_size)
state_dict = torch.load(cfg_u.checkpoint_start, map_location='cpu')
model_u.load_state_dict(state_dict, strict=False)
model_u = model_u.to(device).eval()

val_transforms, _, _ = get_transforms((cfg_u.img_size,) * 2, mean=mean, std=std)
query_dataset = U1652DatasetEval(
    data_folder=cfg_u.query_folder_test, mode='query', transforms=val_transforms)
gallery_dataset = U1652DatasetEval(
    data_folder=cfg_u.gallery_folder_test, mode='gallery',
    transforms=val_transforms, sample_ids=query_dataset.get_sample_ids(), gallery_n=-1)

query_loader   = DataLoader(query_dataset,   batch_size=128, num_workers=4, shuffle=False, pin_memory=True)
gallery_loader = DataLoader(gallery_dataset, batch_size=128, num_workers=4, shuffle=False, pin_memory=True)

print('Extracting features...')
q_feats, q_ids = predict(cfg_u, model_u, query_loader)
g_feats, g_ids = predict(cfg_u, model_u, gallery_loader)
print(f'Query: {q_feats.shape}  Gallery: {g_feats.shape}')

In [ ]:
# Find successful and failed pairs
# Similarity matrix
sim_matrix = (q_feats @ g_feats.T).cpu().numpy()   # (N_query, N_gallery)
q_ids_np = q_ids.cpu().numpy()
g_ids_np = g_ids.cpu().numpy()

successful, failed = [], []

for i in range(len(q_ids_np)):
    sims = sim_matrix[i]
    top1_idx  = int(np.argmax(sims))
    top1_sim  = sims[top1_idx]
    top1_id   = g_ids_np[top1_idx]
    true_id   = q_ids_np[i]

    if top1_id == true_id and len(successful) < 5:
        successful.append({'q_idx': i, 'g_idx': top1_idx, 'sim': top1_sim,
                           'id': true_id})
    elif top1_id != true_id and len(failed) < 3:
        # find correct gallery index
        correct_idxs = np.where(g_ids_np == true_id)[0]
        if len(correct_idxs) > 0:
            failed.append({'q_idx': i, 'g_idx_top1': top1_idx, 'g_idx_correct': correct_idxs[0],
                           'sim': top1_sim, 'id': true_id, 'top1_id': top1_id})

print(f'Successful pairs: {len(successful)}')
for p in successful:
    print(f"  id={p['id']:04d}  sim={p['sim']:.4f}")

print(f'Failed pairs: {len(failed)}')
for p in failed:
    print(f"  true={p['id']:04d}  retrieved={p['top1_id']:04d}  sim={p['sim']:.4f}")

In [ ]:
# Build path lookup from dataset
# U1652DatasetEval stores file paths in .images (query mode) and .images (gallery mode)
# Inspect how paths are stored
print(type(query_dataset))
print(dir(query_dataset))

In [ ]:
# Adapt based on the attribute names found above
# Common patterns: .samples, .images, .img_paths, .data_list
# Example (adjust attribute name if different):
q_paths = query_dataset.samples    # list of (path, label) or just paths
g_paths = gallery_dataset.samples  # same

# If samples is list of (path, id) tuples:
def get_path(samples, idx):
    entry = samples[idx]
    return entry[0] if isinstance(entry, (list, tuple)) else entry

# Preview
print(get_path(q_paths, 0))
print(get_path(g_paths, 0))

## 7. GradCAM — University-1652


In [ ]:
gradcam = GradCAMExtractor(model_u)

for i, pair in enumerate(successful, 1):
    q_path = get_path(q_paths, pair['q_idx'])
    g_path = get_path(g_paths, pair['g_idx'])

    q_t = load_image(q_path).to(device)
    g_t = load_image(g_path).to(device)

    q_cam, g_cam = gradcam.generate_pair_cam(q_t, g_t)

    sim = pair['sim']
    save_path = f"{RESULTS_DIR}/gradcam/u1652_successful_{i:02d}.png"
    show_gradcam(q_t, g_t, q_cam, g_cam,
                 title=f"GradCAM — Successful #{i} | id={pair['id']:04d} | sim={sim:.4f}",
                 save_path=save_path)

gradcam.remove_hooks()

In [ ]:
gradcam = GradCAMExtractor(model_u)

for i, pair in enumerate(failed, 1):
    q_path  = get_path(q_paths, pair['q_idx'])
    g_path  = get_path(g_paths, pair['g_idx_top1'])   # wrong gallery

    q_t = load_image(q_path).to(device)
    g_t = load_image(g_path).to(device)

    q_cam, g_cam = gradcam.generate_pair_cam(q_t, g_t)

    save_path = f"{RESULTS_DIR}/gradcam/u1652_failed_{i:02d}.png"
    show_gradcam(q_t, g_t, q_cam, g_cam,
                 title=f"GradCAM — Failed #{i} | true={pair['id']:04d} retrieved={pair['top1_id']:04d} | sim={pair['sim']:.4f}",
                 save_path=save_path)

gradcam.remove_hooks()

## 8. Occlusion Sensitivity — University-1652

Slower than GradCAM: ~(H/stride)² forward passes per image.  
With `patch_size=64, stride=16, img_size=384`: about 576 forward passes per image.


In [ ]:
PATCH_SIZE = 64
STRIDE     = 16
FAITH_STEPS = 10

occ = OcclusionSensitivity(model_u, device=torch.device(device))

for i, pair in enumerate(successful, 1):
    q_path = get_path(q_paths, pair['q_idx'])
    g_path = get_path(g_paths, pair['g_idx'])

    q_t = load_image(q_path).to(device)
    g_t = load_image(g_path).to(device)

    print(f'Computing sensitivity for successful #{i}...')
    q_map, g_map = occ.compute_pair_sensitivity(q_t, g_t, PATCH_SIZE, STRIDE)

    with torch.no_grad():
        g_emb = model_u(g_t)
        q_emb = model_u(q_t)

    q_faith = occ.compute_faithfulness(q_t, g_emb, q_map, steps=FAITH_STEPS)
    g_faith = occ.compute_faithfulness(g_t, q_emb, g_map, steps=FAITH_STEPS)

    save_path = f"{RESULTS_DIR}/occlusion/u1652_successful_{i:02d}.png"
    show_occlusion(q_t, g_t, q_map, g_map,
                   q_faith=q_faith, g_faith=g_faith,
                   title=f"Occlusion — Successful #{i} | id={pair['id']:04d} | sim={pair['sim']:.4f}",
                   save_path=save_path)

In [ ]:
for i, pair in enumerate(failed, 1):
    q_path = get_path(q_paths, pair['q_idx'])
    g_path = get_path(g_paths, pair['g_idx_top1'])

    q_t = load_image(q_path).to(device)
    g_t = load_image(g_path).to(device)

    print(f'Computing sensitivity for failed #{i}...')
    q_map, g_map = occ.compute_pair_sensitivity(q_t, g_t, PATCH_SIZE, STRIDE)

    with torch.no_grad():
        g_emb = model_u(g_t)
        q_emb = model_u(q_t)

    q_faith = occ.compute_faithfulness(q_t, g_emb, q_map, steps=FAITH_STEPS)
    g_faith = occ.compute_faithfulness(g_t, q_emb, g_map, steps=FAITH_STEPS)

    save_path = f"{RESULTS_DIR}/occlusion/u1652_failed_{i:02d}.png"
    show_occlusion(q_t, g_t, q_map, g_map,
                   q_faith=q_faith, g_faith=g_faith,
                   title=f"Occlusion — Failed #{i} | true={pair['id']:04d} retrieved={pair['top1_id']:04d}",
                   save_path=save_path)

## 9. XAI Pair Selection — VIGOR

Same process as University-1652 but with VIGOR's VigorDatasetEval.


In [ ]:
# Run for same-area (repeat for cross-area by changing same_area=False and checkpoint)
VIGOR_SAME_AREA = True
ckpt_vigor = CKPT_VIGOR_SAME if VIGOR_SAME_AREA else CKPT_VIGOR_CROSS

cfg_v = VigorConfig(same_area=VIGOR_SAME_AREA, checkpoint_start=ckpt_vigor)

model_v = TimmModel(cfg_v.model, pretrained=True, img_size=cfg_v.img_size)
data_config_v = model_v.get_config()
mean_v, std_v = data_config_v['mean'], data_config_v['std']

state_dict = torch.load(cfg_v.checkpoint_start, map_location='cpu')
model_v.load_state_dict(state_dict, strict=False)
model_v = model_v.to(device).eval()

img_size_v = cfg_v.img_size
image_size_sat = (img_size_v, img_size_v)
new_width  = img_size_v * 2
new_height = int(((1024 - 2 * cfg_v.ground_cutting) / 2048) * new_width)
img_size_ground = (new_height, new_width)

sat_transforms, ground_transforms = get_transforms_val(
    image_size_sat, img_size_ground, mean=mean_v, std=std_v,
    ground_cutting=cfg_v.ground_cutting)

ref_dataset_v = VigorDatasetEval(
    data_folder=VIGOR_DATA, split='test', img_type='reference',
    same_area=VIGOR_SAME_AREA, transforms=sat_transforms)
q_dataset_v = VigorDatasetEval(
    data_folder=VIGOR_DATA, split='test', img_type='query',
    same_area=VIGOR_SAME_AREA, transforms=ground_transforms)

ref_loader_v = DataLoader(ref_dataset_v, batch_size=128, num_workers=4, shuffle=False, pin_memory=True)
q_loader_v   = DataLoader(q_dataset_v,   batch_size=128, num_workers=4, shuffle=False, pin_memory=True)

print('Extracting VIGOR features...')
q_feats_v, q_ids_v = predict(cfg_v, model_v, q_loader_v)
g_feats_v, g_ids_v = predict(cfg_v, model_v, ref_loader_v)
print(f'Query: {q_feats_v.shape}  Gallery: {g_feats_v.shape}')

In [ ]:
# Pair selection for VIGOR
# Note: VIGOR evaluation uses different matching logic (multiple positives per query)
# For XAI we just pick top-1 match and check if it's in the positive set
sim_v = (q_feats_v @ g_feats_v.T).cpu().numpy()
q_ids_v_np = q_ids_v.cpu().numpy()
g_ids_v_np = g_ids_v.cpu().numpy()

vigor_successful, vigor_failed = [], []

for i in range(len(q_ids_v_np)):
    sims    = sim_v[i]
    top1_idx = int(np.argmax(sims))
    top1_sim = sims[top1_idx]
    top1_id  = g_ids_v_np[top1_idx]
    true_id  = q_ids_v_np[i]

    if top1_id == true_id and len(vigor_successful) < 5:
        vigor_successful.append({'q_idx': i, 'g_idx': top1_idx, 'sim': top1_sim})
    elif top1_id != true_id and len(vigor_failed) < 3:
        correct_idxs = np.where(g_ids_v_np == true_id)[0]
        if len(correct_idxs) > 0:
            vigor_failed.append({'q_idx': i, 'g_idx_top1': top1_idx,
                                  'g_idx_correct': correct_idxs[0], 'sim': top1_sim})

print(f'VIGOR successful: {len(vigor_successful)}')
print(f'VIGOR failed:     {len(vigor_failed)}')

## 10. GradCAM — VIGOR


In [ ]:
# Path lookup for VIGOR datasets
print(dir(q_dataset_v))  # inspect to find path attribute

In [ ]:
# Adjust attribute name based on above output
# VigorDatasetEval likely stores paths in idx2ground_path / idx2sat_path
def get_vigor_query_path(dataset, idx):
    return dataset.idx2ground_path[idx]

def get_vigor_gallery_path(dataset, idx):
    return dataset.idx2sat_path[idx]

gradcam_v = GradCAMExtractor(model_v)

split_tag = 'same' if VIGOR_SAME_AREA else 'cross'

for i, pair in enumerate(vigor_successful, 1):
    q_path = get_vigor_query_path(q_dataset_v,   pair['q_idx'])
    g_path = get_vigor_gallery_path(ref_dataset_v, pair['g_idx'])

    q_t = load_image(q_path, img_size=img_size_v).to(device)
    g_t = load_image(g_path, img_size=img_size_v).to(device)

    q_cam, g_cam = gradcam_v.generate_pair_cam(q_t, g_t)

    save_path = f"{RESULTS_DIR}/gradcam/vigor_{split_tag}_successful_{i:02d}.png"
    show_gradcam(q_t, g_t, q_cam, g_cam,
                 title=f"GradCAM — VIGOR {split_tag} Successful #{i} | sim={pair['sim']:.4f}",
                 save_path=save_path)

gradcam_v.remove_hooks()

In [ ]:
gradcam_v = GradCAMExtractor(model_v)

for i, pair in enumerate(vigor_failed, 1):
    q_path = get_vigor_query_path(q_dataset_v,     pair['q_idx'])
    g_path = get_vigor_gallery_path(ref_dataset_v, pair['g_idx_top1'])

    q_t = load_image(q_path, img_size=img_size_v).to(device)
    g_t = load_image(g_path, img_size=img_size_v).to(device)

    q_cam, g_cam = gradcam_v.generate_pair_cam(q_t, g_t)

    save_path = f"{RESULTS_DIR}/gradcam/vigor_{split_tag}_failed_{i:02d}.png"
    show_gradcam(q_t, g_t, q_cam, g_cam,
                 title=f"GradCAM — VIGOR {split_tag} Failed #{i} | sim={pair['sim']:.4f}",
                 save_path=save_path)

gradcam_v.remove_hooks()

## 11. Occlusion Sensitivity — VIGOR


In [ ]:
occ_v = OcclusionSensitivity(model_v, device=torch.device(device))

for i, pair in enumerate(vigor_successful, 1):
    q_path = get_vigor_query_path(q_dataset_v,   pair['q_idx'])
    g_path = get_vigor_gallery_path(ref_dataset_v, pair['g_idx'])

    q_t = load_image(q_path, img_size=img_size_v).to(device)
    g_t = load_image(g_path, img_size=img_size_v).to(device)

    print(f'VIGOR occlusion successful #{i}...')
    q_map, g_map = occ_v.compute_pair_sensitivity(q_t, g_t, PATCH_SIZE, STRIDE)

    with torch.no_grad():
        g_emb = model_v(g_t)
        q_emb = model_v(q_t)

    q_faith = occ_v.compute_faithfulness(q_t, g_emb, q_map, steps=FAITH_STEPS)
    g_faith = occ_v.compute_faithfulness(g_t, q_emb, g_map, steps=FAITH_STEPS)

    save_path = f"{RESULTS_DIR}/occlusion/vigor_{split_tag}_successful_{i:02d}.png"
    show_occlusion(q_t, g_t, q_map, g_map,
                   q_faith=q_faith, g_faith=g_faith,
                   title=f"Occlusion — VIGOR {split_tag} Successful #{i}",
                   save_path=save_path)

## 12. Analysis

Key questions to answer:

1. **GradCAM vs Occlusion Sensitivity** — Do they highlight the same regions? Where do they disagree?
2. **Successful vs Failed** — Does the model attend to semantically meaningful regions when it's correct? What does it attend to when it fails?
3. **University-1652 vs VIGOR** — Does attention pattern differ between drone↔satellite and street↔satellite?
4. **Faithfulness** — How quickly does similarity drop as important regions are masked? Steeper = more faithful explanation.


In [ ]:
# Results summary — fill in after running all sections
results = {
    'University-1652': {
        'R@1':  None,  # fill from Section 3
        'R@5':  None,
        'R@10': None,
        'AP':   None,
    },
    'VIGOR Same-Area': {
        'R@1':  None,  # fill from Section 4
        'R@5':  None,
        'R@10': None,
    },
    'VIGOR Cross-Area': {
        'R@1':  None,
        'R@5':  None,
        'R@10': None,
    },
}

for dataset, metrics in results.items():
    print(f'{dataset}:')
    for k, v in metrics.items():
        print(f'  {k}: {v}')

## Next Steps

- Fill in results table above after running all eval sections
- Write analysis comparing GradCAM vs Occlusion heatmaps
- Write analysis comparing successful vs failed matches
- Write analysis comparing University-1652 vs VIGOR attention patterns
- Export figures from `xai_results/` for use in the report
- Save results to Drive: `%cp -r /content/Sample4Geo/xai_results /content/drive/MyDrive/vision-datasets/`


In [ ]:
# Save all XAI results back to Drive
import shutil
save_to = f'{DRIVE_ROOT}/xai_results'
if os.path.exists(save_to):
    shutil.rmtree(save_to)
shutil.copytree(RESULTS_DIR, save_to)
print(f'Results saved to Drive: {save_to}')